# Model Training: Vehicle Insurance Cross-Sell

Life cycle of this notebook:
- Import data and required packages
- Load the data and build features
- Temporal train / validation / test split
- Preprocessing pipeline (leakage-safe)
- Baseline and candidate models
- Hyper-parameter tuning with time-aware cross-validation
- Choose the best model
- Calibration and the decision threshold
- Final evaluation on the sealed test window (run once)
- Fairness check, feature importance, saving the model, scoring the unlabelled file

**Before running:** in your activated venv, install the modelling libraries (and add them to `requirements.txt`):

```
pip install scikit-learn lightgbm joblib
pip install shap        # optional, only for the SHAP cell
```

Naming used throughout:
- `train_df`, `val_df`, `test_df`: the labelled file split by policy start date. `test_df` is the **sealed** test window and is used only once, at the end.
- `score_df`: the separate unlabelled file (`test_data_cross-sell.csv`) that gets the final predictions.

## 1) Setup

### 1.1 Import data and required packages

In [ ]:
import os
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

warnings.filterwarnings("ignore")

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (OneHotEncoder, StandardScaler, KBinsDiscretizer,
                                   FunctionTransformer, TargetEncoder)
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.metrics import (average_precision_score, roc_auc_score, brier_score_loss, log_loss,
                             precision_recall_curve, roc_curve)
from sklearn.calibration import calibration_curve
from sklearn.isotonic import IsotonicRegression

try:
    from lightgbm import LGBMClassifier
except ImportError:
    raise ImportError("LightGBM is not installed. In your venv terminal run: pip install lightgbm")

Settings you can change. Keep the defaults for the first full run.

In [ ]:
RANDOM_STATE = 42
N_TRAIN = None        # set to e.g. 60000 for a quick, smaller training run while testing code
N_ITER = 10           # number of random hyper-parameter combinations to try
VALUE_TO_COST = 10    # ASSUMPTION: one sale is worth 10x the cost of contacting a customer (replace with real figures)
np.random.seed(RANDOM_STATE)

### 1.2 Load the data and build features

The same steps as in the EDA notebook: read `id` as text, parse the text dates with an explicit format, and derive `age`, `vehicle_age` and one interaction flag.

The flag `damage_not_insured` marks customers with past vehicle damage who are not yet insured. The EDA (4.4.2) showed that this segment holds about 97% of all interested customers. Age and vehicle age are measured **at the policy start date**, so no feature uses information from after the policy began.

In [ ]:
FMT = "%A, %B %d, %Y"

def make_features(df):
    out = df.copy()
    for c in ["Customer_Date_of_Birth", "Life_policy_start_date"]:
        out[c] = pd.to_datetime(out[c], format=FMT)
    out["age"] = (out["Life_policy_start_date"] - out["Customer_Date_of_Birth"]).dt.days / 365.25
    out["vehicle_age"] = out["Life_policy_start_date"].dt.year - out["Vehicle_Year_of_Manufacture"]
    out["damage_not_insured"] = ((out["Vehicle_Damage"] == "Yes") & (out["Previously_Insured"] == 0)).astype(int)
    return out

raw_train = pd.read_csv("data/train_data_cross-sell.csv", dtype={"id": str})
train_all = make_features(raw_train)
print(train_all.shape)
train_all.head()

### 1.3 Temporal train / validation / test split

Train on the earliest 70% of policies, validate on the next 15% and keep the latest 15% as the **sealed test window**. This matches the preview in the EDA (4.5.1). The rows are sorted by date, which the time-aware cross-validation in 1.10 needs.

In [ ]:
train_all = train_all.sort_values("Life_policy_start_date", kind="stable").reset_index(drop=True)
dates = train_all["Life_policy_start_date"]
cut_1, cut_2 = dates.quantile(0.70), dates.quantile(0.85)

train_df = train_all[dates <= cut_1]
val_df = train_all[(dates > cut_1) & (dates <= cut_2)]
test_df = train_all[dates > cut_2]

if N_TRAIN:
    train_df = train_df.sample(N_TRAIN, random_state=RANDOM_STATE).sort_values("Life_policy_start_date", kind="stable")

summary = pd.DataFrame({
    "rows": [len(train_df), len(val_df), len(test_df)],
    "response_rate": [train_df["Response"].mean(), val_df["Response"].mean(), test_df["Response"].mean()],
    "first_date": [train_df["Life_policy_start_date"].min(), val_df["Life_policy_start_date"].min(), test_df["Life_policy_start_date"].min()],
    "last_date": [train_df["Life_policy_start_date"].max(), val_df["Life_policy_start_date"].max(), test_df["Life_policy_start_date"].max()],
}, index=["train", "validation", "test (sealed)"]).round(4)
summary

### 1.4 Preparing X and y

In [ ]:
FEATURES = ["age", "vehicle_age", "Annual_Premium", "Driving_License", "Previously_Insured", "damage_not_insured",
            "Gender", "Vehicle_Damage", "Customer_Residence_Sub_County", "Agent_name"]
TARGET = "Response"

X_train, y_train = train_df[FEATURES], train_df[TARGET].to_numpy()
X_val, y_val = val_df[FEATURES], val_df[TARGET].to_numpy()
X_test, y_test = test_df[FEATURES], test_df[TARGET].to_numpy()

print("Features used:", FEATURES)
print("Not used: id (unreliable), raw dates (replaced by age) and Vehicle_Year_of_Manufacture (replaced by vehicle_age)")
X_train.head()

### 1.5 Preprocessing pipeline (leakage-safe)

Everything that learns from the data (scalers, encoders) sits **inside** the pipeline, so it is fitted on training rows only.

- **Sub-county and agent** use `TargetEncoder`, which encodes each category with smoothing and fits its encoding with internal cross-fitting, so a row's own label never leaks into its own feature. Unseen categories fall back to the overall response rate.
- **Gender and `Vehicle_Damage`** are one-hot encoded.
- **Tree models** use the numeric columns as they are.
- **Logistic Regression** gets age bands (the age effect is hump-shaped), a log-transformed premium (it is skewed), and scaled inputs.

The target-encoded features score slightly better on the training rows than on new rows, because the final encoder has seen those rows. So read the train scores in 1.7 as optimistic and rely on validation.

In [ ]:
def make_preprocessor(kind="tree", use_agent=True):
    te_cols = ["Customer_Residence_Sub_County"] + (["Agent_name"] if use_agent else [])
    flags = ["Driving_License", "Previously_Insured", "damage_not_insured"]
    ohe = OneHotEncoder(drop="if_binary", handle_unknown="ignore", sparse_output=False)
    te = TargetEncoder(smooth="auto", cv=5, random_state=RANDOM_STATE)

    if kind == "tree":
        parts = [
            ("num", "passthrough", ["age", "vehicle_age", "Annual_Premium"]),
            ("flags", "passthrough", flags),
            ("cat", ohe, ["Gender", "Vehicle_Damage"]),
            ("te", te, te_cols),
        ]
    else:
        parts = [
            ("age_bins", KBinsDiscretizer(n_bins=6, encode="onehot-dense", strategy="quantile"), ["age"]),
            ("premium", Pipeline([("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
                                  ("scale", StandardScaler())]), ["Annual_Premium"]),
            ("vehicle_age", StandardScaler(), ["vehicle_age"]),
            ("flags", "passthrough", flags),
            ("cat", ohe, ["Gender", "Vehicle_Damage"]),
            ("te", Pipeline([("te", te), ("scale", StandardScaler())]), te_cols),
        ]
    return ColumnTransformer(parts, verbose_feature_names_out=False)

# quick check: fit on the training rows only and look at the result
check = make_preprocessor("tree").fit(X_train, y_train)
print("Transformed training shape:", check.transform(X_train).shape)
print("Feature names:", list(check.get_feature_names_out()))

### 1.6 Evaluation function

Accuracy is not used. The function reports:
- **PR-AUC** (primary metric, because only about 12% of customers respond)
- **ROC-AUC**
- **Recall@top10%** and **Lift@top10%**: if only the 10% of customers with the highest scores are contacted, what share of all interested customers is reached, and how many times better is that than contacting 10% at random?
- **Brier score** and **log-loss**, which measure how good the probabilities are, not only the ranking.

In [ ]:
def evaluate_model(y_true, p, top_frac=0.10):
    y_true = np.asarray(y_true)
    p = np.asarray(p, dtype=float)
    n_top = int(np.ceil(top_frac * len(p)))
    order = np.argsort(-p, kind="stable")
    captured = y_true[order[:n_top]]
    return {
        "PR_AUC": average_precision_score(y_true, p),
        "ROC_AUC": roc_auc_score(y_true, p),
        "Recall@top10%": captured.sum() / y_true.sum(),
        "Lift@top10%": captured.mean() / y_true.mean(),
        "Brier": brier_score_loss(y_true, p),
        "LogLoss": log_loss(y_true, np.clip(p, 1e-6, 1 - 1e-6)),
    }

def threshold_report(y_true, p, thr):
    y_true = np.asarray(y_true)
    sel = np.asarray(p) >= thr
    tp = int(((y_true == 1) & sel).sum()); fp = int(((y_true == 0) & sel).sum()); fn = int(((y_true == 1) & ~sel).sum())
    precision = tp / max(tp + fp, 1); recall = tp / max(tp + fn, 1)
    return {"threshold": round(float(thr), 4), "contacted_share": sel.mean(), "precision": precision,
            "recall": recall, "F1": 2 * precision * recall / max(precision + recall, 1e-12),
            "TP": tp, "FP": fp, "FN": fn}

### 1.7 Baseline and candidate models

The first row is the **majority-class baseline**: it gives every customer the same probability, so its PR-AUC equals the base rate and its lift is about 1. Every other model is judged by how far it beats it.

Candidates (as in the proposal): Logistic Regression as the interpretable baseline, Random Forest as a comparison, and LightGBM as the expected best model. Two LightGBM variants test two questions raised by the EDA:
- **class-weighted LightGBM:** does re-weighting the rare class help, and what does it do to the probabilities?
- **LightGBM without agent:** how much does the model depend on `Agent_name`, the possible proxy?

In [ ]:
pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

def lgbm(**kw):
    params = dict(n_estimators=300, learning_rate=0.05, num_leaves=31, min_child_samples=100,
                  subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                  random_state=RANDOM_STATE, n_jobs=-1, verbose=-1, importance_type="gain")
    params.update(kw)
    return LGBMClassifier(**params)

def make_models():
    return {
        "Majority-class baseline": Pipeline([("pre", make_preprocessor("linear")), ("model", DummyClassifier(strategy="prior"))]),
        "Logistic Regression": Pipeline([("pre", make_preprocessor("linear")), ("model", LogisticRegression(C=1.0, max_iter=2000))]),
        "Random Forest": Pipeline([("pre", make_preprocessor("tree")), ("model", RandomForestClassifier(
            n_estimators=150, max_depth=14, min_samples_leaf=50, n_jobs=-1, random_state=RANDOM_STATE))]),
        "LightGBM": Pipeline([("pre", make_preprocessor("tree")), ("model", lgbm())]),
        "LightGBM (class-weighted)": Pipeline([("pre", make_preprocessor("tree")), ("model", lgbm(scale_pos_weight=pos_weight))]),
        "LightGBM (no agent)": Pipeline([("pre", make_preprocessor("tree", use_agent=False)), ("model", lgbm())]),
    }

fitted, val_probs, rows = {}, {}, []

def register(name, pipe):
    p_tr = pipe.predict_proba(X_train)[:, 1]
    p_va = pipe.predict_proba(X_val)[:, 1]
    fitted[name], val_probs[name] = pipe, p_va
    rows[:] = [r for r in rows if r["model"] != name]
    rows.append({"model": name, "split": "train", **evaluate_model(y_train, p_tr)})
    rows.append({"model": name, "split": "validation", **evaluate_model(y_val, p_va)})

for name, pipe in make_models().items():
    t0 = time.time()
    pipe.fit(X_train, y_train)
    register(name, pipe)
    va = [r for r in rows if r["model"] == name and r["split"] == "validation"][0]
    print(f"{name:28s} fitted in {time.time() - t0:6.1f}s | validation PR-AUC {va['PR_AUC']:.4f}")

### 1.8 Results

In [ ]:
def results_table():
    res = pd.DataFrame(rows)
    val = res[res["split"] == "validation"].set_index("model").drop(columns="split")
    val["train_PR_AUC"] = res[res["split"] == "train"].set_index("model")["PR_AUC"]
    val["PR_AUC_gap"] = val["train_PR_AUC"] - val["PR_AUC"]
    base = val.loc["Majority-class baseline", "PR_AUC"]
    val["PR_AUC_x_baseline"] = val["PR_AUC"] / base
    return val.sort_values("PR_AUC", ascending=False).round(4)

results_table()

How to read it: `PR_AUC_x_baseline` is the lift over the baseline. `PR_AUC_gap` (train minus validation) shows overfitting: a large gap means the model memorised the training rows (train scores for target-encoded features are slightly optimistic, see 1.5).

#### 1.8.1 Precision-recall, ROC and gains curves on the validation window

In [ ]:
def gains_curve(y_true, p):
    order = np.argsort(-np.asarray(p), kind="stable")
    cum = np.cumsum(np.asarray(y_true)[order]) / np.sum(y_true)
    return np.arange(1, len(p) + 1) / len(p), cum

plot_models = [m for m in val_probs if m != "Majority-class baseline"]
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
for m in plot_models:
    prec, rec, _ = precision_recall_curve(y_val, val_probs[m])
    axes[0].plot(rec, prec, label=m)
    fpr, tpr, _ = roc_curve(y_val, val_probs[m])
    axes[1].plot(fpr, tpr, label=m)
    frac, cum = gains_curve(y_val, val_probs[m])
    axes[2].plot(frac, cum, label=m)
axes[0].axhline(y_val.mean(), color="grey", linestyle="--", label="baseline")
axes[0].set(xlabel="Recall", ylabel="Precision", title="Precision-recall (validation)")
axes[1].plot([0, 1], [0, 1], color="grey", linestyle="--")
axes[1].set(xlabel="False positive rate", ylabel="True positive rate", title="ROC (validation)")
axes[2].plot([0, 1], [0, 1], color="grey", linestyle="--", label="random targeting")
axes[2].set(xlabel="Share of customers contacted (highest scores first)", ylabel="Share of interested customers reached",
            title="Cumulative gains (validation)")
for ax in axes:
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

### 1.9 Hyper-parameter tuning with time-aware cross-validation

LightGBM is expected to be the best model, so it gets tuned. `TimeSeriesSplit` keeps every validation fold **later in time** than its training rows, and the target encoder is re-fitted inside each fold, so there is no look-ahead or encoding leakage. The score is PR-AUC (`average_precision`). The validation window is not used for tuning.

In [ ]:
param_dist = {
    "model__n_estimators": [200, 400, 600],
    "model__learning_rate": [0.02, 0.05, 0.1],
    "model__num_leaves": [15, 31, 63],
    "model__min_child_samples": [50, 100, 200, 400],
    "model__subsample": [0.7, 0.85, 1.0],
    "model__colsample_bytree": [0.6, 0.8, 1.0],
    "model__reg_lambda": [0, 1, 10],
}

tune_pipe = Pipeline([("pre", make_preprocessor("tree")), ("model", lgbm())])
search = RandomizedSearchCV(
    tune_pipe, param_dist, n_iter=N_ITER, scoring="average_precision",
    cv=TimeSeriesSplit(n_splits=3), random_state=RANDOM_STATE, n_jobs=1, refit=True, verbose=1,
)
t0 = time.time()
search.fit(X_train, y_train)
print(f"Search finished in {time.time() - t0:.0f}s")
print("Best cross-validated PR-AUC:", round(search.best_score_, 4))
print("Best parameters:", {k.replace('model__', ''): v for k, v in search.best_params_.items()})

register("LightGBM (tuned)", search.best_estimator_)
results_table()

### 1.10 Choose the best model

In [ ]:
tbl = results_table()
candidates = tbl.drop(index=["Majority-class baseline", "LightGBM (no agent)"])   # the no-agent run is a diagnostic, not a candidate
best_name = candidates["PR_AUC"].idxmax()
final_pipe = fitted[best_name]

print("Best model on the validation window:", best_name)
print(candidates.loc[best_name].to_dict())
print()
agent_effect = tbl.loc["LightGBM", "PR_AUC"] - tbl.loc["LightGBM (no agent)", "PR_AUC"]
print(f"Agent audit: dropping Agent_name changes validation PR-AUC by {-agent_effect:+.4f} (LightGBM, same settings)")

**Agent audit:** if the PR-AUC barely moves without `Agent_name`, the agent column adds little beyond the other features, and removing it is cheap protection against the proxy risk. If it drops a lot, agent is carrying real information (or acting as a proxy), and that must be explained in the report.

#### 1.10.1 Decile table of the chosen model (validation)

In [ ]:
def decile_table(y_true, p):
    d = pd.DataFrame({"y": np.asarray(y_true), "p": np.asarray(p)})
    d["decile"] = pd.qcut(d["p"].rank(method="first", ascending=False), 10, labels=range(1, 11))
    t = d.groupby("decile", observed=True).agg(customers=("y", "size"), interested=("y", "sum"),
                                              response_rate=("y", "mean"), avg_predicted=("p", "mean"))
    t["lift"] = t["response_rate"] / d["y"].mean()
    t["cumulative_capture"] = t["interested"].cumsum() / d["y"].sum()
    return t.round(4)

decile_table(y_val, val_probs[best_name])

### 1.11 Calibration and the decision threshold

The output that matters is a **probability**, so it should mean what it says: among customers scored 0.30, about 30% should be interested. The reliability curve checks this on the validation window. If the curve departs from the diagonal, an isotonic regression fitted on the validation predictions maps the scores onto honest probabilities.

In [ ]:
p_val_raw = val_probs[best_name]
frac_pos, mean_pred = calibration_curve(y_val, p_val_raw, n_bins=10, strategy="quantile")

plt.figure(figsize=(6, 6))
plt.plot(mean_pred, frac_pos, marker="o", label=best_name)
plt.plot([0, 1], [0, 1], linestyle="--", color="grey", label="perfectly calibrated")
plt.xlabel("Mean predicted probability")
plt.ylabel("Observed share interested")
plt.title("Reliability curve (validation)")
plt.legend()
plt.show()

iso = IsotonicRegression(out_of_bounds="clip").fit(p_val_raw, y_val)
print("Validation Brier score, raw:", round(brier_score_loss(y_val, p_val_raw), 5))

**Two thresholds, two purposes:**
- **Profit threshold** (outreach decisions): contact a customer when the expected sale is worth more than the cost of contacting them. `VALUE_TO_COST` is an assumption (set in 1.1), so the sensitivity table shows how the answer changes with it. Replace it with ICEA LION's real figures, or justify the value in the report.
- **F1 threshold** (the 0/1 `Response` labels for the submission file): balances precision and recall.

Both are chosen on the validation window, never on the sealed test window.

In [ ]:
p_val_cal = iso.predict(p_val_raw)
grid = np.linspace(0.01, 0.95, 95)

def profit_per_customer(y_true, p, thr, ratio):
    sel = p >= thr
    tp = ((np.asarray(y_true) == 1) & sel).sum()
    return (ratio * tp - sel.sum()) / len(p)      # in units of the cost of one contact

profits = np.array([profit_per_customer(y_val, p_val_cal, t, VALUE_TO_COST) for t in grid])
thr_profit = grid[profits.argmax()]

prec, rec, thr_pr = precision_recall_curve(y_val, p_val_cal)
f1 = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-12)
thr_f1 = thr_pr[f1.argmax()]

print(f"Profit threshold (value/cost = {VALUE_TO_COST}): {thr_profit:.3f}")
print(f"F1 threshold: {thr_f1:.3f}")
print()
print("Validation window at each threshold:")
display(pd.DataFrame({"profit threshold": threshold_report(y_val, p_val_cal, thr_profit),
                      "F1 threshold": threshold_report(y_val, p_val_cal, thr_f1)}).T.round(4))

print("Sensitivity of the profit threshold to the assumed value/cost ratio (validation):")
sens = []
for ratio in [3, 5, 10, 20, 50]:
    pr = np.array([profit_per_customer(y_val, p_val_cal, t, ratio) for t in grid])
    thr = grid[pr.argmax()]
    rep = threshold_report(y_val, p_val_cal, thr)
    sens.append({"value_to_cost": ratio, "threshold": round(thr, 3), "contacted_share": rep["contacted_share"],
                 "recall": rep["recall"], "precision": rep["precision"], "profit_per_customer": pr.max()})
pd.DataFrame(sens).round(4)

### 1.12 Final evaluation on the sealed test window (run once)

This is the only cell that looks at `test_df`. Run it once, after the choices above are final, and report what it shows even if it is lower than validation.

Isotonic calibration turns the scores into steps with many ties, which can lower PR-AUC slightly compared with the raw scores. Use the calibrated column when you quote probabilities, and say so if the raw column ranks marginally better.

In [ ]:
p_test_raw = final_pipe.predict_proba(X_test)[:, 1]
p_test_cal = iso.predict(p_test_raw)

final_scores = pd.DataFrame({
    "raw probabilities": evaluate_model(y_test, p_test_raw),
    "calibrated probabilities": evaluate_model(y_test, p_test_cal),
}).T.round(4)
print("Final model:", best_name)
display(final_scores)

print("Majority-class baseline on the same window: PR-AUC =", round(y_test.mean(), 4), "(the base rate), lift about 1")
print()
print("At the chosen thresholds (selected on validation):")
display(pd.DataFrame({"profit threshold": threshold_report(y_test, p_test_cal, thr_profit),
                      "F1 threshold": threshold_report(y_test, p_test_cal, thr_f1)}).T.round(4))

print("Decile table (test window, calibrated probabilities):")
display(decile_table(y_test, p_test_cal))

#### 1.12.1 Gains and calibration on the test window

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
frac, cum = gains_curve(y_test, p_test_cal)
axes[0].plot(frac, cum, label=best_name)
axes[0].plot([0, 1], [0, 1], linestyle="--", color="grey", label="random targeting")
axes[0].set(xlabel="Share of customers contacted", ylabel="Share of interested customers reached", title="Cumulative gains (test)")
axes[0].legend()

for label, p in [("raw", p_test_raw), ("calibrated", p_test_cal)]:
    fp, mp = calibration_curve(y_test, p, n_bins=10, strategy="quantile")
    axes[1].plot(mp, fp, marker="o", label=label)
axes[1].plot([0, 1], [0, 1], linestyle="--", color="grey")
axes[1].set(xlabel="Mean predicted probability", ylabel="Observed share interested", title="Reliability (test)")
axes[1].legend()
plt.tight_layout()
plt.show()

### 1.13 Fairness check by gender

The proposal commits to reporting performance by gender. Interested customers are more common among men (13.8% vs 10.5%, EDA 4.4.1), so a model can pick more men even if it treats both groups consistently. The table separates three things: how well the model ranks within each group (PR-AUC, ROC-AUC), whether its probabilities are honest within each group (mean predicted vs actual), and how often each group is selected at the profit threshold.

In [ ]:
fair_rows = []
for g in ["Female", "Male"]:
    mask = (test_df["Gender"] == g).to_numpy()
    ev = evaluate_model(y_test[mask], p_test_cal[mask])
    fair_rows.append({
        "group": g, "customers": int(mask.sum()), "actual_rate": y_test[mask].mean(),
        "mean_predicted": p_test_cal[mask].mean(), "PR_AUC": ev["PR_AUC"], "ROC_AUC": ev["ROC_AUC"],
        "Recall@top10%": ev["Recall@top10%"],
        "selected_at_profit_thr": (p_test_cal[mask] >= thr_profit).mean(),
        "recall_at_profit_thr": threshold_report(y_test[mask], p_test_cal[mask], thr_profit)["recall"],
    })
fair = pd.DataFrame(fair_rows).set_index("group").round(4)
display(fair)
print("Selection-rate gap (male minus female):", round(fair.loc["Male", "selected_at_profit_thr"] - fair.loc["Female", "selected_at_profit_thr"], 4))
print("Recall gap (male minus female):", round(fair.loc["Male", "recall_at_profit_thr"] - fair.loc["Female", "recall_at_profit_thr"], 4))

### 1.14 Which features drive the model?

In [ ]:
pre = final_pipe.named_steps["pre"]
model = final_pipe.named_steps["model"]
names = list(pre.get_feature_names_out())

if hasattr(model, "feature_importances_"):
    imp = pd.Series(model.feature_importances_, index=names)
elif hasattr(model, "coef_"):
    imp = pd.Series(np.abs(model.coef_[0]), index=names)
else:
    imp = pd.Series(dtype=float)

if len(imp):
    imp = (imp / imp.sum()).sort_values(ascending=False)
    plt.figure(figsize=(8, 5))
    imp.head(12)[::-1].plot(kind="barh")
    plt.title(f"Feature importance, {best_name} (share of total)")
    plt.tight_layout()
    plt.show()
    display(imp.head(12).round(4))

Feature importance says which inputs the model uses, not which way they push. SHAP gives the direction per customer (optional cell, needs `pip install shap`).

In [ ]:
try:
    import shap
    sample = X_val.sample(min(5000, len(X_val)), random_state=RANDOM_STATE)
    Xs = pre.transform(sample)
    sv = shap.TreeExplainer(model).shap_values(Xs)
    if isinstance(sv, list):
        sv = sv[1]
    elif getattr(sv, "ndim", 2) == 3:
        sv = sv[:, :, 1]
    shap.summary_plot(sv, Xs, feature_names=names, show=False)
    plt.tight_layout()
    plt.show()
except ImportError:
    print("SHAP is not installed. Run: pip install shap   (then re-run this cell)")
except Exception as e:
    print("SHAP could not run for this model:", type(e).__name__, e)

### 1.15 Save the model

In [ ]:
os.makedirs("../artifacts", exist_ok=True)
joblib.dump({
    "pipeline": final_pipe, "calibrator": iso, "model_name": best_name, "features": FEATURES,
    "threshold_profit": float(thr_profit), "threshold_f1": float(thr_f1),
}, "../artifacts/notebook_model.joblib")
print("Saved ../artifacts/notebook_model.joblib")

This file is for the notebook only. The production pipeline (`model_trainer.py`) will save the preprocessing, model and calibrator as one artefact for FastAPI, and MLflow will log the experiments there.

### 1.16 Predict the unlabelled file

The case study asks for a copy of `test_data_cross-sell.csv` with `Response` filled in. The 0/1 labels use the **F1 threshold**. The second file also carries the calibrated probability of each customer, which is the output that matters for targeting.

In [ ]:
raw_score = pd.read_csv("data/test_data_cross-sell.csv", dtype={"id": str})
score_df = make_features(raw_score)

p_score = iso.predict(final_pipe.predict_proba(score_df[FEATURES])[:, 1])

submission = raw_score.copy()
submission["Response"] = (p_score >= thr_f1).astype(int)
with_prob = submission.assign(Response_probability=np.round(p_score, 4))

submission.to_csv("../artifacts/test_data_cross-sell_predictions.csv", index=False)
with_prob.to_csv("../artifacts/test_data_cross-sell_predictions_with_probabilities.csv", index=False)

print("Rows scored:", len(submission))
print("Predicted interested (Response = 1):", int(submission['Response'].sum()), f"({submission['Response'].mean():.1%})")
print("Average predicted probability:", round(float(p_score.mean()), 4), "| training response rate:", round(float(train_all['Response'].mean()), 4))
with_prob.head()

## 2) Findings (fill in after running)

- **Baseline:** majority class has PR-AUC = ___ (the base rate) and lift about 1.
- **Best model:** ___ with validation PR-AUC ___ (___ times the baseline) and ROC-AUC ___.
- **Overfitting:** PR-AUC gap between train and validation: ___ for the best model, ___ for Random Forest.
- **Top 10%:** contacting the top 10% of scored customers reaches ___% of interested customers (lift ___), on the sealed test window.
- **Agent audit:** dropping `Agent_name` changes validation PR-AUC by ___, so agent (adds / does not add) much beyond the other features.
- **Calibration:** Brier score ___ raw vs ___ calibrated on the test window.
- **Threshold:** at a value-to-cost ratio of ___ the profit threshold is ___ and contacts ___% of customers, reaching ___% of interested customers.
- **Fairness:** the male-minus-female selection-rate gap is ___ and the recall gap is ___.
- **Top drivers:** ___, ___ and ___ (feature importance and SHAP).
- **Final test score (sealed window):** PR-AUC ___, ROC-AUC ___, Lift@top10% ___.